In [0]:
Data Profiling

import pandas as pd
import numpy as np
     

caso_practico_ventas = spark.read.table("workspace.default.caso_practico_ventas").toPandas()
     

caso_practico_ventas
     
Se confirma que la tabla tiene un total de 123 filas por 9 columnas


display(caso_practico_ventas.head(10))
display(caso_practico_ventas.tail(5))
     
Diagnóstico: Al realizar la carga e inspección inicial del dataset, los tipos de datos inferidos automáticamente no coinciden en su totalidad con los tipos esperados según el diccionario de datos. También se puede observar que la columna id_transaccion contiene transacciones repetidas como TXN-10011


# Conteo absoluto de nulos 
nulos_abs = caso_practico_ventas.isnull().sum()
     

# Porcentaje de nulos 
nulos_pct = (caso_practico_ventas.isnull().sum() / len(caso_practico_ventas)) * 100
     

# Consolidado en un DataFrame 
caso_practico_ventas_nulos = pd.DataFrame({'Total Nulos': nulos_abs, 'Porcentaje (%)': nulos_pct.round(2)}) 
print(caso_practico_ventas_nulos)
     
Al revisar el resultado, encontramos datos faltantes únicamente en 3 de las 9 columnas. Las demás variables (id_transaccion, fecha, id_cliente, categoria_producto, metodo_pago y region) vienen completas al 100%.


#Calculo de media, mediana, desviación estándar, mínimo y máximo para cantidad y precio_unitario.
columnas_num = ['cantidad', 'precio_unitario']
estadisticas = caso_practico_ventas[columnas_num].agg(['mean', 'median', 'std', 'min', 'max']).T
estadisticas.columns = ['Media', 'Mediana', 'Desviación Estándar', 'Mínimo', 'Máximo']
display(estadisticas)
     
Al revisar las métricas estadísticas calculadas, se identifica lo siguiente:

Unidades vendidas (cantidad): Presenta un comportamiento muy regular. La media y la mediana coinciden exactamente en 2 unidades. Casi todas las ventas consisten en compras pequeñas de entre 1 y 4 unidades, teniendo un mínimo de 1 y un máximo de 10 unidades.

Precios (precio_unitario): Se nota un sesgo marcado hacia arriba debido a un valor extremo. La mediana es de 
186.41 con una desviación estándar de óá
9,999.00, que corresponde a una transacción anómala que deberá corregirse más adelante.

Data Cleaning

# 1. Identificar cuántas filas duplicadas existen 
cant_duplicados = caso_practico_ventas.duplicated(subset=['id_transaccion']).sum() 
print(f"Total de registros duplicados: {cant_duplicados}")
     

# 2. Eliminar registros duplicados conservando el primero 
caso_practico_ventas_clean = caso_practico_ventas.drop_duplicates(subset=['id_transaccion'])
     

# 3. Verificar la nueva cantidad de filas 
print(f"Filas iniciales: {len(caso_practico_ventas)} | Filas tras eliminar duplicados: {len(caso_practico_ventas_clean)}")
     
se identifican 3 registros duplicados exactos dentro del conjunto de datos original de 123 filas.

Tras aplicar la depuración, el dataset quedó conformado por un total de 120 registros.


# 1. Ver la frecuencia de valores antes de corregir 
print("Antes de estandarizar:")
print(caso_practico_ventas_clean['metodo_pago'].value_counts()) 

     

# 2. Reemplazar la variante inconsistente 
caso_practico_ventas_clean['metodo_pago'] = caso_practico_ventas_clean['metodo_pago'].replace({'Tarjeta Credito': 'Tarjeta de Crédito'})
     

# 3. Ver la frecuencia corregida 
print("\nDespués de estandarizar:") 
print(caso_practico_ventas_clean['metodo_pago'].value_counts())
     
Luego de unificar las etiquetas "Tarjeta Credito" (27 registros) y "Tarjeta de Crédito" (26 registros) se tiene un total de 53 transacciones.


# 1. Calcular la mediana de 'cantidad' y la moda de 'satisfaccion\_cliente' 
mediana_cantidad = caso_practico_ventas_clean['cantidad'].median()
moda_satisfaccion = caso_practico_ventas_clean['satisfaccion_cliente'].mode()[0] 
print(f"Mediana calculada para cantidad: {mediana_cantidad}") 
print(f"Moda calculada para satisfacción: {moda_satisfaccion}") 


     

# 2. Imputar (rellenar) los valores nulos 
caso_practico_ventas_clean['cantidad'] = caso_practico_ventas_clean['cantidad'].fillna(mediana_cantidad)
caso_practico_ventas_clean['satisfaccion_cliente'] = caso_practico_ventas_clean['satisfaccion_cliente'].fillna(moda_satisfaccion) 
     

# 3. Verificar que ya no existan nulos en estas dos columnas 
print("\Nulos restantes en cantidad:", caso_practico_ventas_clean['cantidad'].isnull().sum()) 
print("Nulos restantes en satisfacción:", caso_practico_ventas_clean['satisfaccion_cliente'].isnull().sum())
     
Unidades vendidas (cantidad): Al ser una variable cuantitativa discreta, los 5 valores nulos identificados se imputaron utilizando la mediana del conjunto de datos (2.0 unidades).

Satisfacción del cliente (satisfaccion_cliente): Al tratarse de una escala ordinal (de 1 a 5), los 2 valores nulos se completaron mediante la moda (4.0), que representa la calificación más habitual otorgada por los compradores.

Tras este proceso, ambas columnas quedaron con el 100% de sus datos completos (0 nulos).


# 1. Convertir el valor atípico de 9999.0 a NaN para no afectar el promedio 
caso_practico_ventas_clean['precio_unitario'] = caso_practico_ventas_clean['precio_unitario'].replace(9999.0, np.nan) 

     

# 2. Calcular la media de precio por categoría (sin incluir el outlier) e imputar 
caso_practico_ventas_clean['precio_unitario'] = caso_practico_ventas_clean.groupby('categoria_producto')['precio_unitario'].transform( lambda x: x.fillna(x.mean()) ) 

     

# 3. Verificar que no queden nulos ni valores atípicos 
print("Nulos restantes en precio_unitario:", caso_practico_ventas_clean['precio_unitario'].isnull().sum()) 
print("Precio máximo corregido:", caso_practico_ventas_clean['precio_unitario'].max())
     
Luego de esta correción, el dataset quedó completamente depurado (120 registros únicos, 0 nulos en todas las variables y valores de precios consistentes).

Transformación e Ingeniería de Atributos

# 1. Columna derivada del importe total 
caso_practico_ventas_clean['monto_total'] =  caso_practico_ventas_clean['cantidad'] *  caso_practico_ventas_clean['precio_unitario'] 

     

# 2. Conversión temporal y extracción de día y mes 
caso_practico_ventas_clean['fecha'] = pd.to_datetime( caso_practico_ventas_clean['fecha'])
caso_practico_ventas_clean['dia_semana'] =  caso_practico_ventas_clean['fecha'].dt.day_name()
caso_practico_ventas_clean['mes'] =  caso_practico_ventas_clean['fecha'].dt.month_name()
     

# 3. Discretización en rangos de consumo 
def categorizar_monto(monto):
    if monto < 100:
        return 'Bajo'
    elif monto <= 500:
        return 'Medio'
    else:
        return 'Alto'

caso_practico_ventas_clean['rango_monto'] = caso_practico_ventas_clean['monto_total'].apply(categorizar_monto)

# Visualizar primeras filas con los nuevos atributos 
display(caso_practico_ventas_clean[['id_transaccion', 'monto_total', 'dia_semana', 'mes', 'rango_monto']].head())
     
monto_total: Calculado como cantidad * precio_unitario. Presenta una mediana de 
218.62 (máximo de $5,000.00).
Variables temporales (dia_semana y mes): Extraídas automáticamente tras convertir la columna fecha a formato de fecha.
Categorización del consumo (rango_monto): Clasificación de las ventas en tres segmentos:
Bajo (< $100): 69 transacciones (57.5%).
Medio (
500):43 transacciones (35.8%).
Alto (>$500): 8 transacciones (6.7%).
Agregación y Resumen de Datos

# 1. Agrupamiento por Categoría 
resumen_cat = caso_practico_ventas_clean.groupby('categoria_producto').agg( total_ingresos=('monto_total', 'sum'), promedio_unidades=('cantidad', 'mean') ).round(2) 


     

# 2. Tabla Dinámica (Región vs Categoría) 
tabla_dinamica = pd.pivot_table( caso_practico_ventas_clean, values='monto_total', index='region', columns='categoria_producto', aggfunc='sum', fill_value=0 ).round(2)

     

# 3. Satisfacción por Método de Pago 
satisfaccion_pago = caso_practico_ventas_clean.groupby('metodo_pago')['satisfaccion_cliente'].mean().round(2)
display(resumen_cat)
display(tabla_dinamica)
display(satisfaccion_pago)
     
Rendimiento por Categoría: Hogar lidera los ingresos con 
7,788.69), que además registra el volumen medio de compra más alto (2.33 unidades/venta). Deportes obtuvo la menor facturación ($1,895.42).
Distribución Regional: La tabla dinámica muestra concentraciones clave de facturación: la región Este lidera en Hogar (ó
4,096.00).
Satisfacción del Cliente: El pago en Efectivo obtiene la mejor calificación de satisfacción (4.08), mientras que la Tarjeta de Crédito registra la menor puntuación promedio (3.51).
Selección, Filtrado y Muestreo

# 1. Filtrado Multicriterio: Región Norte y Satisfacción >= 4 
filtro_norte_sat = caso_practico_ventas_clean[ (caso_practico_ventas_clean['region'] == 'Norte') & (caso_practico_ventas_clean['satisfaccion_cliente'] >= 4) ] 
     

# 2. Muestreo Estratificado (10% por Región) 
muestra_estratificada = caso_practico_ventas_clean.groupby('region', group_keys=False).apply( lambda x: x.sample(frac=0.10, random_state=42) ) 
     

# Mostrar resultados 
print(f"Registros filtrados (Norte & Sat >= 4): {len(filtro_norte_sat)}")
display(filtro_norte_sat.head()) 
print(f"\Tamaño de la muestra estratificada: {len(muestra_estratificada)}")
display(muestra_estratificada)
     
Filtrado Multicriterio: Se aislaron 12 transacciones pertenecientes a la región Norte con un nivel de satisfacción igual o superior a 4.0. Este subconjunto permite analizar el perfil de las ventas más exitosas en dicha zona.
Muestreo Estratificado: Se extrajo una muestra representativa del 10% del dataset (11 registros en total) manteniendo la proporción de ventas según cada region (3 del Norte y 2 de cada una de las demás regiones), lo que permite realizar pruebas ágiles sin perder la representatividad geográfica.